# Writing your own quality function

The quality functions of `pysubgroup` are ordinary Python classes. A new
measure takes a few lines and then works with every search algorithm, with
filters, with `to_dataframe` and with the significance tests. This notebook
builds four quality functions step by step:

1. an **unbounded** measure: it only scores subgroups (beam search, or
   exhaustive search without pruning),
2. a **bounded** measure: an optimistic estimate lets exhaustive searches prune,
3. a **batched** measure: it scores all refinements of a search node in one
   pass (fast on the CPU, and on a GPU with `device="cuda"`),
4. a measure on a **target of its own**: weighted rows.

As in the first notebook, the data is the **Adult** census dataset from OpenML.

In [ ]:
import time

import numpy as np
from sklearn.datasets import fetch_openml

import pysubgroup as sd

adult = fetch_openml("adult", version=2, as_frame=True)
data = adult.frame
# fnlwgt is a census sampling weight; it is not a feature (used in part 4).
features = data.drop(columns=["class", "fnlwgt"])
conditions = sd.generate_conditions(features, n_bins=5)

target = sd.BinaryTarget().fit(data, "class", ">50K")
print(len(conditions), "conditions;", target.n_positives_, "of", target.n_samples_, "rows positive")

## The protocol

A **target** is fitted once on the data and stores per-row information as
positional arrays. `BinaryTarget` stores

- `positives_`: a Boolean array, `True` for the rows with `class == ">50K"`,
- `n_positives_` and `n_samples_`: the counts in the whole dataset.

A **quality function** only keeps its parameters. The search calls
`evaluate(coverage, target)`, where `coverage` is a Boolean array with one
entry per row (by position, not by DataFrame index) marking the rows of the
subgroup. The method returns a float, larger is better.

Subclass `sd.QualityFunction` and set `target_type` to the target class the
measure needs. The helper `self._mask(coverage, target)` checks the target type
and the mask and returns the mask as a NumPy array.

## 1. An unbounded measure: the log odds ratio

The odds ratio compares the odds of the target inside the subgroup with the
odds outside it. From the 2x2 contingency table

|              | positive | negative |
|--------------|----------|----------|
| subgroup     | `tp`     | `fp`     |
| complement   | `fn`     | `tn`     |

we compute `log((tp + c)(tn + c) / ((fp + c)(fn + c)))`. The correction `c = 0.5`
(Haldane-Anscombe) keeps the value finite when a cell is empty.

In [ ]:
class LogOddsRatioQF(sd.QualityFunction):
    # Log odds ratio of the target in the subgroup vs. its complement.

    target_type = sd.BinaryTarget

    def __init__(self, correction: float = 0.5):
        self.correction = correction

    def evaluate(self, coverage, target) -> float:
        mask = self._mask(coverage, target)
        size = np.count_nonzero(mask)
        tp = np.count_nonzero(mask & target.positives_)
        fp = size - tp
        fn = target.n_positives_ - tp
        tn = target.n_samples_ - target.n_positives_ - fp
        c = self.correction
        return float(np.log((tp + c) * (tn + c) / ((fp + c) * (fn + c))))


qf = LogOddsRatioQF()
everyone = np.ones(len(data), dtype=bool)
married = (data["relationship"] == "Husband").to_numpy()
print(f"all rows: {qf.evaluate(everyone, target):.3f}")
print(f"husbands: {qf.evaluate(married, target):.3f}")

The odds ratio ignores the size of a subgroup, so very small subgroups with
only positives would win. A `MinSizeFilter` restricts the search to subgroups
with at least 500 rows. Without an optimistic estimate an exhaustive search
cannot prune, so we use beam search here.

In [ ]:
task = sd.SubgroupDiscoveryTask(
    data, target, conditions, LogOddsRatioQF(), result_set_size=5, depth=3,
    filters=[sd.MinSizeFilter(500)],
)
results = sd.BeamSearch(beam_width=20).execute(task)
sd.to_dataframe(results, data, target, ["size", "target_share", "odds_ratio"]).round(3)

A new measure is also a column in `to_dataframe`, next to the built-in ones:

In [ ]:
wracc_results = sd.DFS().execute(
    sd.SubgroupDiscoveryTask(data, target, conditions, sd.WRAccQF(), result_set_size=5, depth=2)
)
sd.to_dataframe(
    wracc_results, data, target, ["size", "target_share"],
    measures={"log odds ratio": LogOddsRatioQF()},
).round(3)

## 2. A bounded measure: the linear cost measure

Exhaustive searches (DFS, best-first, OPUS, Apriori, SD-Map*, ...) skip a
subgroup and all its refinements when no refinement can enter the result set.
For that, a measure subclasses `sd.BoundedQualityFunction` and adds
`optimistic_estimate(coverage, target)`: an **upper bound on the quality of
every subset** of the subgroup's rows. Refinements cover subsets, so the bound
holds for all of them.

The linear cost measure (a form of the linear cost metric in Fürnkranz and
Flach, *ROC 'n' rule learning*, MLJ 2005) rewards covered
positives and charges a cost `cost` per covered negative:
`(tp - cost * fp) / n`. A subset covers at most the `tp` positives and at
least no negatives, so its quality is at most `tp / n`. That is the
optimistic estimate; it is reached by the subset of the subgroup's positives.

In [ ]:
class LinearCostQF(sd.BoundedQualityFunction):
    # (tp - cost * fp) / n: covered positives minus the cost of covered negatives.

    target_type = sd.BinaryTarget

    def __init__(self, cost: float):
        self.cost = cost

    @staticmethod
    def _counts(mask, target):
        tp = np.count_nonzero(mask & target.positives_)
        return tp, np.count_nonzero(mask) - tp

    def evaluate(self, coverage, target) -> float:
        tp, fp = self._counts(self._mask(coverage, target), target)
        return (tp - self.cost * fp) / target.n_samples_

    def optimistic_estimate(self, coverage, target) -> float:
        tp, _ = self._counts(self._mask(coverage, target), target)
        return tp / target.n_samples_

A wrong bound makes the search silently miss subgroups, so test it. A quick
check draws random subsets of some subgroups and compares their quality with
the subgroup's estimate:

In [ ]:
def check_bound(qf, target, masks, trials=200, seed=0):
    # Raise if a random subset of a mask scores above the mask's optimistic estimate.
    rng = np.random.default_rng(seed)
    for mask in masks:
        bound = qf.optimistic_estimate(mask, target)
        rows = np.flatnonzero(mask)
        for _ in range(trials):
            subset = np.zeros_like(mask)
            subset[rng.choice(rows, rng.integers(1, len(rows) + 1), replace=False)] = True
            quality = qf.evaluate(subset, target)
            assert quality <= bound + 1e-12, (quality, bound)
    print("bound holds on", len(masks) * trials, "random subsets")


masks = sd.utils.subgroup_masks(wracc_results, data)
check_bound(LinearCostQF(cost=0.3), target, masks)

A useful sanity check: with `cost = P / (N - P)` (positives over negatives in
the dataset), the linear cost measure is proportional to WRAcc, since
`WRAcc = (1 - P/N) * (tp - cost * fp) / n`. Both measures must then find the
same subgroups, in the same order.

In [ ]:
cost = target.n_positives_ / (target.n_samples_ - target.n_positives_)


def search(qf, depth=3, **options):
    task = sd.SubgroupDiscoveryTask(
        data, target, conditions, qf, result_set_size=10, depth=depth, **options
    )
    start = time.perf_counter()
    found = sd.DFS().execute(task)
    return found, time.perf_counter() - start


linear, seconds = search(LinearCostQF(cost))
wracc, _ = search(sd.WRAccQF())
print(f"linear cost measure: {seconds:.1f} s")
print("same subgroups as WRAcc:", [rule for _, rule in linear] == [rule for _, rule in wracc])
sd.to_dataframe(linear, data, target, ["size", "target_share"]).round(4)

How much does pruning save? The wrapper below hides the optimistic estimate
(it is a plain `QualityFunction`) and counts the evaluations. Depth 2 keeps
the unpruned search short.

In [ ]:
class Counting(sd.QualityFunction):
    # Delegates to another measure without its optimistic estimate; counts the calls.

    def __init__(self, qf):
        self.qf = qf
        self.target_type = qf.target_type
        self.calls = 0

    def evaluate(self, coverage, target) -> float:
        self.calls += 1
        return self.qf.evaluate(coverage, target)


class CountingBounded(Counting, sd.BoundedQualityFunction):
    def optimistic_estimate(self, coverage, target) -> float:
        return self.qf.optimistic_estimate(coverage, target)


for wrapper in (Counting(LinearCostQF(cost)), CountingBounded(LinearCostQF(cost))):
    _, seconds = search(wrapper, depth=2)
    print(f"{type(wrapper).__name__:16} {wrapper.calls:6} evaluations, {seconds:.2f} s")

## 3. A batched measure

Each node of a search has many refinements, one per remaining condition.
Evaluating them one by one builds a row mask per refinement. A measure with
`batchable = True` instead receives the statistics of all refinements at once,
computed from bit-packed row sets (with numba on the CPU, or with CuPy on a
GPU). `evaluate_many(statistics, target)` and `optimistic_estimate_many`
return one value per refinement. The `statistics` object (`ChildStatistics`)
offers

- `statistics.sizes`: the number of rows of each refinement,
- `statistics.count(rows)`: the number of rows inside a Boolean row array,
- `statistics.sum(values)`: the sum of a per-row array (one or two dimensions),
- `statistics.maximum(values)`: the largest value of a per-row array.

Pass arrays stored on the target (such as `target.positives_`), not new
arrays: the statistics are cached by array, so the quality function and the
optimistic estimate share one pass. The linear cost measure needs only sizes
and positive counts:

In [ ]:
class BatchedLinearCostQF(LinearCostQF):
    batchable = True

    def _counts_many(self, statistics, target):
        self._check_target(target)
        tp = statistics.count(target.positives_)
        return tp, statistics.sizes - tp

    def evaluate_many(self, statistics, target):
        tp, fp = self._counts_many(statistics, target)
        return (tp - self.cost * fp) / target.n_samples_

    def optimistic_estimate_many(self, statistics, target):
        tp, _ = self._counts_many(statistics, target)
        return tp / target.n_samples_


scalar, scalar_seconds = search(BatchedLinearCostQF(cost), depth=5, batched=False)
batched, batched_seconds = search(BatchedLinearCostQF(cost), depth=5)
print(f"one at a time: {scalar_seconds:.1f} s, batched: {batched_seconds:.1f} s")
print("same results:", scalar == batched)

The batched methods must return the same values as the scalar ones (up to
floating-point rounding of sums). `SubgroupDiscoveryTask(batched=None)`, the
default, batches whenever the algorithm, the quality function and all pruning
filters support it; `batched=True` raises an error if they do not.

## 4. A target of its own: weighted rows

The census weight `fnlwgt` says how many people in the population a row stands
for. A weighted WRAcc counts people instead of rows:
`W_sg / W * (Wpos_sg / W_sg - Wpos / W)`, with the total weight `W` and the
weight `Wpos` of the positives (`_sg` inside the subgroup).

A new target subclasses `sd.BaseTarget`, or an existing target to keep its
quality functions: a subclass of `sd.BinaryTarget` still works with WRAcc,
the chi-squared test and the binary filters and plots. `fit` calls the base
class, which validates the data and sets `n_samples_`, and stores positional
arrays. `attributes_` names the target columns. `_fit_arguments` returns the
arguments of `fit` after the data, so `refit` (used by the holdout and
randomization tests of `sd.significance`) can fit the same target on other
rows.

Removing all negatives from a subgroup gives the optimistic estimate
`Wpos_sg / W * (1 - Wpos / W)`, as for the unweighted measure.

In [ ]:
class WeightedBinaryTarget(sd.BinaryTarget):
    # Binary target whose rows carry nonnegative weights.

    def fit(self, data, attribute, value, weights):
        super().fit(data, attribute, value)
        self.attributes_ = (attribute, weights)
        self.weights_ = data[weights].to_numpy(dtype=float)
        self.positive_weights_ = np.where(self.positives_, self.weights_, 0.0)
        self.total_weight_ = float(self.weights_.sum())
        self.positive_share_ = float(self.positive_weights_.sum()) / self.total_weight_
        return self

    def _fit_arguments(self):
        return (self.attributes_[0], self.value_, self.attributes_[1])


class WeightedWRAccQF(sd.BoundedQualityFunction):
    # WRAcc over row weights: W_sg / W * (Wpos_sg / W_sg - Wpos / W).

    target_type = WeightedBinaryTarget
    batchable = True

    @staticmethod
    def _quality(weight, positive_weight, target):
        with np.errstate(divide="ignore", invalid="ignore"):
            share = positive_weight / weight
        return weight / target.total_weight_ * (share - target.positive_share_)

    def _bound(self, positive_weight, target):
        return positive_weight / target.total_weight_ * (1 - target.positive_share_)

    def _weights(self, mask, target):
        return target.weights_[mask].sum(), target.positive_weights_[mask].sum()

    def evaluate(self, coverage, target) -> float:
        weight, positive_weight = self._weights(self._mask(coverage, target), target)
        return float(self._quality(weight, positive_weight, target))

    def optimistic_estimate(self, coverage, target) -> float:
        _, positive_weight = self._weights(self._mask(coverage, target), target)
        return float(self._bound(positive_weight, target))

    def evaluate_many(self, statistics, target):
        self._check_target(target)
        weight = statistics.sum(target.weights_)
        return self._quality(weight, statistics.sum(target.positive_weights_), target)

    def optimistic_estimate_many(self, statistics, target):
        self._check_target(target)
        return self._bound(statistics.sum(target.positive_weights_), target)


weighted = WeightedBinaryTarget().fit(data, "class", ">50K", "fnlwgt")
print(f"share of >50K: {target.n_positives_ / target.n_samples_:.4f} of rows, "
      f"{weighted.positive_share_:.4f} of people")
check_bound(WeightedWRAccQF(), weighted, masks)

The new target works with the searches as any other. `to_dataframe` shows the
statistics of binary targets (unweighted); the weighted share and the
unweighted WRAcc go in as extra columns. The weighted measure prefers
subgroups whose rows stand for many people.

In [ ]:
task = sd.SubgroupDiscoveryTask(
    data, weighted, conditions, WeightedWRAccQF(), result_set_size=10, depth=3
)
class WeightedShareQF(sd.QualityFunction):
    # Weighted share of the target in the subgroup.

    target_type = WeightedBinaryTarget

    def evaluate(self, coverage, target) -> float:
        mask = self._mask(coverage, target)
        return target.positive_weights_[mask].sum() / target.weights_[mask].sum()


weighted_results = sd.DFS().execute(task)
sd.to_dataframe(
    weighted_results, data, weighted, ["size", "target_share"],
    measures={"WRAcc (rows)": sd.WRAccQF(), "weighted share": WeightedShareQF()},
).round(4)

## Checklist

- **Return floats, larger is better.** For measures where smaller is better
  (such as p-values), return the negation or `1 - p`. Return `nan` for empty
  subgroups if the measure is undefined there, and `-inf` for subgroups that
  should never be reported.
- **Optimistic estimates must hold for every subset of the rows**, not only for
  the refinements a search happens to visit; check them on random subsets as
  above, or prove them. A loose bound is correct but prunes less; a bound that
  is too small loses results without any warning.
- **Batched and scalar methods must agree.** Compare `batched=True` with
  `batched=False` on a few searches (`tests/test_batched_evaluation.py` does
  this for the built-in measures).
- **Keep per-row arrays on the target** and compute them in `fit`, so the
  quality function is cheap and the batched statistics are cached.
- **Targets that are refitted** (holdout and randomization tests) need
  `_fit_arguments`.
- **Directions:** the built-in measures take
  `direction="positive" | "negative" | "both"`; `sd.targets.base.check_direction`
  validates it.

The built-in measures in `pysubgroup/targets/` follow the same protocol and
are good templates: `binary.py` for counts, `numeric.py` for measures that
need the ordered target values, `emm/` for model-based targets.